# SK하이닉스 종합 분석

주가, 네이버 뉴스, 증권사 리포트(KIS → 네이버 증권 → 한경컨센서스), LLM 분석을 실행합니다.

In [4]:
# SK하이닉스 종합 분석: 주가 + 네이버 뉴스 + LLM
# 필요한 패키지: pykrx, requests, pandas, python-dotenv
import os
import datetime as dt
import json
import contextlib
import io
import re
import requests
import time
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from email.utils import parsedate_to_datetime
from html import unescape
from IPython.display import display
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from pykrx import stock

_dotenv_paths = []
for _base_path in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    _dotenv_paths.extend([
        _base_path / ".env",
        _base_path / "notebooks" / ".env",
    ])
if "__file__" in globals():
    _dotenv_paths.extend([
        Path(__file__).resolve().with_name(".env"),
        Path(__file__).resolve().parents[1] / "notebooks" / ".env",
    ])
for _dotenv_path in _dotenv_paths:
    if _dotenv_path.is_file():
        load_dotenv(_dotenv_path, override=True)

STOCK_NAME = "SK하이닉스"
STOCK_TICKER = "000660"
LOOKBACK_DAYS = 30
NEWS_COUNT = 20
REPORT_COUNT = 20

KIS_BASE_URL = os.getenv("KIS_BASE_URL", "https://openapi.koreainvestment.com:9443")
KIS_TOKEN_URL = f"{KIS_BASE_URL}/oauth2/tokenP"
KIS_RESEARCH_URL = f"{KIS_BASE_URL}/uapi/domestic-stock/v1/quotations/research-info"




In [5]:
def _clean_html(value: str) -> str:
    return unescape(value.replace("<b>", "").replace("</b>", "")).strip()


def get_stock_history(ticker: str, days: int = 30) -> pd.DataFrame:
    """최근 영업일 주가와 거래량을 조회한다."""
    end = dt.date.today()
    start = end - dt.timedelta(days=days)
    history = stock.get_market_ohlcv_by_date(
        start.strftime("%Y%m%d"), end.strftime("%Y%m%d"), ticker
    )
    if history.empty:
        raise RuntimeError(f"{ticker} 종목의 주가 데이터를 찾지 못했습니다.")
    return history


def search_stock_news(query: str, display: int = 20) -> pd.DataFrame:
    """네이버 뉴스 검색 API에서 최신 관련 기사를 가져온다."""
    client_id = os.getenv("NAVER_CLIENT_ID")
    client_secret = os.getenv("NAVER_CLIENT_SECRET")
    if not client_id or not client_secret:
        raise RuntimeError(".env에 NAVER_CLIENT_ID와 NAVER_CLIENT_SECRET을 설정하세요.")

    response = requests.get(
        "https://openapi.naver.com/v1/search/news.json",
        headers={
            "X-Naver-Client-Id": client_id,
            "X-Naver-Client-Secret": client_secret,
        },
        params={"query": query, "display": display, "sort": "date"},
        timeout=20,
    )
    if not response.ok:
        raise RuntimeError(f"네이버 뉴스 API 오류 ({response.status_code}): {response.text}")

    rows = []
    for item in response.json().get("items", []):
        published_at = item.get("pubDate")
        rows.append({
            "title": _clean_html(item.get("title", "")),
            "description": _clean_html(item.get("description", "")),
            "link": item.get("link", ""),
            "pubDate": (
                parsedate_to_datetime(published_at)
                if published_at
                else None
            ),
        })
    return pd.DataFrame(rows)


def _number(value):
    """문자열 숫자에서 쉼표·통화기호를 제거해 숫자로 변환한다."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value).replace(",", "").replace("원", "").strip()
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group()) if match else None


def _recommendation(value: str) -> str:
    """증권사별 투자의견을 매수·중립·매도로 통일한다."""
    text = str(value or "").strip().lower().replace(" ", "")
    if any(word in text for word in ("매수", "buy", "outperform", "overweight", "strongbuy")):
        return "매수"
    if any(word in text for word in ("매도", "sell", "underperform", "underweight", "reduce")):
        return "매도"
    if any(word in text for word in ("중립", "보유", "hold", "neutral", "marketperform")):
        return "중립"
    return str(value or "").strip()


def _report_frame(rows) -> pd.DataFrame:
    columns = ["date", "broker", "title", "recommendation", "target_price", "source", "url"]
    frame = pd.DataFrame(rows)
    if frame.empty:
        return pd.DataFrame(columns=columns)
    for column in columns:
        if column not in frame:
            frame[column] = None
    frame["recommendation"] = frame["recommendation"].map(_recommendation)
    frame["target_price"] = frame["target_price"].map(_number)
    return frame[columns].reset_index(drop=True)


KIS_TOKEN_CACHE_PATH = Path(".kis_token_cache.json")


def _kis_access_token(force_refresh: bool = False) -> str:
    app_key = os.getenv("KIS_APP_KEY")
    app_secret = os.getenv("KIS_APP_SECRET")
    if not app_key or not app_secret:
        raise RuntimeError(".env에 KIS_APP_KEY와 KIS_APP_SECRET을 설정하세요.")
    if not force_refresh and KIS_TOKEN_CACHE_PATH.exists():
        try:
            cache = json.loads(KIS_TOKEN_CACHE_PATH.read_text(encoding="utf-8"))
            if (
                cache.get("app_key") == app_key
                and time.time() < cache.get("expires_at", 0)
                and cache.get("access_token")
            ):
                return cache["access_token"]
        except (OSError, json.JSONDecodeError):
            pass

    response = requests.post(
        KIS_TOKEN_URL,
        json={
            "grant_type": "client_credentials",
            "appkey": app_key,
            "appsecret": app_secret,
        },
        timeout=20,
    )
    if not response.ok:
        raise RuntimeError(f"KIS 토큰 발급 오류 ({response.status_code}): {response.text}")
    payload = response.json()
    token = payload.get("access_token")
    if not token:
        raise RuntimeError("KIS 토큰 응답에 access_token이 없습니다.")
    KIS_TOKEN_CACHE_PATH.write_text(
        json.dumps({
            "app_key": app_key,
            "access_token": token,
            "expires_at": time.time() + int(payload.get("expires_in", 86400)) - 300,
        }),
        encoding="utf-8",
    )
    return token


def _get_kis_reports(ticker: str, count: int) -> pd.DataFrame:
    app_key = os.getenv("KIS_APP_KEY")
    app_secret = os.getenv("KIS_APP_SECRET")
    token = _kis_access_token()
    authorization = "Bearer " + token
    response = requests.get(
        KIS_RESEARCH_URL,
        headers={
            "authorization": f"Bearer {token}",
            "appkey": app_key,
            "appsecret": app_secret,
            "tr_id": "FHKST01010900",
            "custtype": "P",
            "content-type": "application/json; charset=utf-8",
        },
        params={"FID_COND_MRKT_DIV_CODE": "J", "FID_INPUT_ISCD": ticker},
        timeout=20,
    )
    if not response.ok:
        raise RuntimeError(f"KIS 종목투자의견 API 오류 ({response.status_code}): {response.text}")
    payload = response.json()
    if str(payload.get("rt_cd", "0")) != "0":
        raise RuntimeError(f"KIS 종목투자의견 API 오류: {payload.get('msg1', payload)}")

    output = payload.get("output1") or payload.get("output") or []
    if isinstance(output, dict):
        output = [output]
    rows = []
    for item in output:
        rows.append({
            "date": item.get("stck_bsop_date") or item.get("date") or item.get("rpt_dt"),
            "broker": (
                item.get("증권사")
                or item.get("brk_name")
                or item.get("broker_name")
                or item.get("invt_opnn_org")
            ),
            "title": item.get("rpt_nm") or item.get("report_title") or item.get("title"),
            "recommendation": (
                item.get("invt_opnn")
                or item.get("invt_opnn_cls")
                or item.get("opinion")
                or item.get("recommendation")
            ),
            "target_price": (
                item.get("목표주가")
                or item.get("stck_tgpr")
                or item.get("target_price")
                or item.get("tgt_prc")
            ),
            "source": "KIS",
            "url": item.get("url") or item.get("report_url"),
        })
    frame = _report_frame(rows)
    if frame.empty:
        raise RuntimeError("KIS 종목투자의견 API가 리포트를 반환하지 않았습니다.")
    return frame.head(count)


def _get_naver_reports(ticker: str, count: int) -> pd.DataFrame:
    url = f"https://finance.naver.com/research/company_list.naver?searchType=itemCode&itemCode={ticker}"
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20,
    )
    response.raise_for_status()
    tables = pd.read_html(response.text)
    rows = []
    for table in tables:
        table.columns = [str(column).strip() for column in table.columns]
        opinion_column = next((c for c in table.columns if "의견" in c), None)
        target_column = next((c for c in table.columns if "목표" in c), None)
        if not opinion_column or not target_column:
            continue
        for _, item in table.iterrows():
            rows.append({
                "date": item.get("날짜") or item.get("작성일"),
                "broker": item.get("증권사"),
                "title": item.get("제목"),
                "recommendation": item.get(opinion_column),
                "target_price": item.get(target_column),
                "source": "네이버 증권",
                "url": url,
            })
    frame = _report_frame(rows)
    if frame.empty:
        raise RuntimeError("네이버 증권 리서치 탭에서 리포트를 찾지 못했습니다.")
    return frame.head(count)


def _get_hankyung_reports(ticker: str, count: int) -> pd.DataFrame:
    url = f"https://consensus.hankyung.com/analysis/search?item_code={ticker}"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
    response.raise_for_status()
    tables = pd.read_html(response.text)
    rows = []
    for table in tables:
        table.columns = [str(column).strip() for column in table.columns]
        opinion_column = next((c for c in table.columns if "의견" in c), None)
        target_column = next((c for c in table.columns if "목표" in c), None)
        if not opinion_column or not target_column:
            continue
        for _, item in table.iterrows():
            rows.append({
                "date": item.get("작성일") or item.get("날짜"),
                "broker": item.get("증권사"),
                "title": item.get("리포트명") or item.get("제목"),
                "recommendation": item.get(opinion_column),
                "target_price": item.get(target_column),
                "source": "한경컨센서스",
                "url": url,
            })
    frame = _report_frame(rows)
    if frame.empty:
        raise RuntimeError("한경컨센서스에서 리포트를 찾지 못했습니다.")
    return frame.head(count)


def get_stock_reports(ticker: str, count: int = 20) -> pd.DataFrame:
    """KIS 종목투자의견을 조회하고 실패하면 네이버, 한경 순으로 폴백한다."""
    if not re.fullmatch(r"\d{6}", ticker):
        raise ValueError("ticker는 6자리 숫자여야 합니다.")

    errors = []
    for source in (_get_kis_reports, _get_naver_reports, _get_hankyung_reports):
        try:
            reports = source(ticker, count)
            if not reports.empty:
                return reports
        except (requests.RequestException, ValueError, RuntimeError, ImportError) as error:
            errors.append(f"{source.__name__}: {error}")
    raise RuntimeError("증권사 리포트를 수집하지 못했습니다. " + " | ".join(errors))


def analyze_with_llm(stock_name: str, ticker: str, history: pd.DataFrame, news: pd.DataFrame) -> str:
    """주가 흐름과 뉴스의 관계를 xAI LLM에 분석시킨다."""
    api_key = os.getenv("XAI_API_KEY")
    if not api_key:
        raise RuntimeError(".env에 XAI_API_KEY를 설정하세요.")

    price = history.copy().reset_index()
    price_text = price.tail(15).to_string(index=False)
    news_text = (
        news[["title", "description", "pubDate", "link"]].to_string(index=False)
        if not news.empty
        else "관련 뉴스 없음"
    )

    system = """너는 한국 주식 리서치 애널리스트다. 제공된 데이터만 근거로 분석하고, 확인된 사실과 해석을 구분하라.
투자 매수·매도 권유나 확정적인 미래 예측은 하지 말라."""
    prompt = f"""다음은 {stock_name}({ticker})의 최근 주가와 네이버 뉴스다.

[주가 데이터]
{price_text}

[관련 뉴스]
{news_text}

아래 형식으로 한국어 종합 분석을 작성해라.
1. 최근 주가 흐름: 기간 수익률, 고점·저점, 거래량 변화
2. 핵심 뉴스 요약: 주가에 영향을 줄 수 있는 뉴스 3~5개
3. 주가 변동 원인: 뉴스와 주가·거래량의 시간적 흐름을 연결한 근거 중심 분석
4. 긍정 요인과 부정 요인
5. 추가 확인할 리스크와 다음 거래일에 관찰할 지표
각 항목은 간결한 문단 또는 bullet로 작성하고, 근거가 부족하면 '판단 유보'라고 표시해라."""

    response = requests.post(
        "https://api.x.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": os.getenv("XAI_MODEL", "grok-4-1-fast-non-reasoning"),
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
            "temperature": 0.2,
        },
        timeout=120,
    )
    if not response.ok:
        raise RuntimeError(f"xAI API 오류 ({response.status_code}): {response.text}")
    return response.json()["choices"][0]["message"]["content"]




In [6]:
price_df = get_stock_history(STOCK_TICKER, LOOKBACK_DAYS)
news_df = search_stock_news(STOCK_NAME, NEWS_COUNT)
try:
    reports_df = get_stock_reports(STOCK_TICKER, REPORT_COUNT)
except RuntimeError as error:
    print(f"증권사 리포트 수집 건너뜀: {error}")
    reports_df = pd.DataFrame(
        columns=["date", "broker", "title", "recommendation", "target_price", "source", "url"]
    )

first_close = float(price_df["종가"].iloc[0])
last_close = float(price_df["종가"].iloc[-1])
period_return = (last_close / first_close - 1) * 100
print(f"종목: {STOCK_NAME} ({STOCK_TICKER})")
print(f"조회기간: {price_df.index.min().date()} ~ {price_df.index.max().date()}")
print(f"최근 종가: {last_close:,.0f}원 | 기간 수익률: {period_return:+.2f}%")
print(f"수집 뉴스: {len(news_df)}건")
report_source = reports_df["source"].iloc[0] if not reports_df.empty else "없음"
print(f"수집 리포트: {len(reports_df)}건 ({report_source})")
display(price_df.tail(10))
display(news_df.head(10))
display(reports_df)

llm_report = analyze_with_llm(STOCK_NAME, STOCK_TICKER, price_df, news_df)
print()
print("===== LLM 종합 분석 =====")
print()
print(llm_report)


증권사 리포트 수집 건너뜀: 증권사 리포트를 수집하지 못했습니다. _get_kis_reports: KIS 종목투자의견 API 오류 (404):  | _get_naver_reports: 네이버 증권 리서치 탭에서 리포트를 찾지 못했습니다. | _get_hankyung_reports: 404 Client Error: Not Found for url: https://consensus.hankyung.com/analysis/search?item_code=000660
종목: SK하이닉스 (000660)
조회기간: 2026-07-27 ~ 2026-08-24
최근 종가: 1,671,000원 | 기간 수익률: -7.98%
수집 뉴스: 20건
수집 리포트: 0건 (없음)


C:\Users\홍준기\AppData\Local\Temp\ipykernel_26480\4101438134.py:199: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


,시가,고가,저가,종가,거래량,등락률
날짜,,,,,,
2026-08-10,1443000,1487000,1397000,1420000,3295283,-0.140647
2026-08-11,1405000,1455000,1373000,1425000,3817655,0.352113
2026-08-12,1456000,1549000,1440000,1504000,4566672,5.543860
2026-08-13,1582000,1634000,1567000,1593000,4680347,5.917553
2026-08-14,1695000,1697000,1626000,1645000,4520990,3.264281
2026-08-18,1736000,1792000,1640000,1662000,5139000,1.033435
2026-08-19,1545000,1559000,1486000,1500000,4223067,-9.747292
2026-08-20,1613000,1720000,1576000,1691000,5452970,12.733333
2026-08-21,1675000,1773000,1669000,1730000,4274294,2.306328


,title,description,link,pubDate
0,[미국 특징주] 엔비디아의 반도체 시장에 바짝 다가서는 우군과 적군들,"앤스로픽은 자체 AI 가속기 개발 계획에 대해 구체적으로 밝히지 않았지만, SK하이...",https://www.newspim.com/news/view/20260825000008,2026-08-25 02:30:00+09:00
1,역대 최대 110조 주주환원에도 -9%… 삼성전자 왜 떨어지나,SK하이닉스가 40조원대 주주환원책 발표 이후 오름세인 것과 대비된다. 시장 기대에...,https://n.news.naver.com/mnews/article/005/000...,2026-08-25 02:08:00+09:00
2,"한투운용, 'ACE 삼성전자SK하이닉스플러스채권혼합50 ETF' 신규 상장",한국투자신탁운용이 'ACE 삼성전자SK하이닉스플러스채권혼합50' ETF를 신규 상장...,http://www.bizwnews.com/news/articleView.html?...,2026-08-25 01:30:00+09:00
3,삼전 8.7% 급락 뒤 美마이크론 -6.8%…외국인 '현물 3.7조 매도·선물 매...,특히 엔비디아보다 마이크론과 SOXL의 낙폭이 크다는 점에서 25일 국내장에서는 삼...,https://www.greened.kr/news/articleView.html?i...,2026-08-25 00:57:00+09:00
4,"‘호남 팹 건설·美 투자 요구’ 이중 부담에… 총수들, 李와 연쇄 독대","美, 호남 반도체 언급하며 압박 미국은 삼성전자와 SK하이닉스 등 국내 반도체 기업...",https://n.news.naver.com/mnews/article/023/000...,2026-08-25 00:52:00+09:00
5,“자사주 소각 빠졌다” 삼성전자 110조 주주환원에도 급락,지난 19일 40조원어치 자사주 전량 소각 방침을 밝힌 SK하이닉스는 사정이 반대다...,https://n.news.naver.com/mnews/article/023/000...,2026-08-25 00:49:00+09:00
6,증시 자사주 소각액 2년새 4.5배로 늘어,현금 배당은 작년 50조 돌파 삼성전자와 SK하이닉스가 잇달아 대규모 주주 환원책을...,https://n.news.naver.com/mnews/article/023/000...,2026-08-25 00:49:00+09:00
7,“어차피 AI 쓸 텐데”… 채용 시장서 사라지는 자소서,"하이닉스, 자소서 대신 면접 강화 소프트뱅크는 자기소개 영상 심사 SK하이닉스는 지...",https://n.news.naver.com/mnews/article/023/000...,2026-08-25 00:35:00+09:00
8,"업비트 운영 두나무, 나스닥 상장 추진",SK하이닉스가 지난달 이 방식으로 나스닥에 상장했다. 업비트를 운영하는 법인이 한국...,https://n.news.naver.com/mnews/article/023/000...,2026-08-25 00:32:00+09:00
9,[단신]하나은행 ‘트래블로그 외화통장’ 10만좌 돌파 外,"■ 한투운용, 반도체주와 채권 절반씩 담은 ETF 출시 한국투자신탁운용은 국내 주요...",https://n.news.naver.com/mnews/article/020/000...,2026-08-25 00:31:00+09:00


,date,broker,title,recommendation,target_price,source,url



===== LLM 종합 분석 =====

1. 최근 주가 흐름  
• 기간(8.3~8.24) 수익률: 1567천 → 1671천 (+6.6%)  
• 고점 1792천(8.18), 저점 1373천(8.11) → 변동폭 30.5%  
• 거래량: 8.3~8.7 500만 주대 → 8.10~8.11 330~380만 주로 감소 후 8.12 이후 450만 주 이상 회복  
• 등락률: 8.6(-10.4%), 8.19(-9.7%), 8.20(+12.7%) 등 ±10%대 급변일 3회 관측  

2. 핵심 뉴스 요약  
• SK하이닉스 40조원 규모 주주환원(자사주 전량 소각) 발표(8.19 공시 이후 보도)  
• 미국 상무장관, 삼성전자·SK하이닉스에 대한 반도체 투자 압박 보도(8.25)  
• 엔비디아 AI 가속기 공급망에 SK하이닉스 메모리 칩 포함 가능성(8.25)  
• ‘ACE 삼성전자SK하이닉스플러스채권혼합50’ ETF 신규 상장(8.25)  
• 광주 군공항 이전 부지에 SK하이닉스 투자설명회 개최(8.24~25)  

3. 주가 변동 원인  
• 8.6~8.7 급락(-15%): 8.5 장중 고점 이후 차익실현 물량 출회, 뚜렷한 촉발 뉴스 미확인  
• 8.12~8.14 반등(+15%): 8.12 거래량 회복과 동반, 8.19 주주환원 공시를 선반영한 매수세 유입 가능성  
• 8.19(-9.7%)→8.20(+12.7%): 8.19 종가 하락 후 8.20 대량 거래(545만 주) 수반한 반등 → 주주환원 기대와 차익매물 소화가 혼재한 단기 변동성  
• 8.24 소폭 하락(-3.4%): 8.25 보도된 美 투자 압박 선반영 가능성 있으나 시간적 선후 관계 불확실  

4. 긍정 요인과 부정 요인  
긍정  
• 40조원 규모 자사주 소각으로 유통 주식 수 감소 기대  
• AI 가속기용 메모리 공급 논의 보도  

부정  
• 美 정부의 추가 투자 요구 리스크  
• 8월 중순 이후 ±10%대 일간 변동성 지속  

5. 추가 확인할 리스크와 다음 거래일 